# Conflict 3: Out-of-Network Amplification vs. Prioritarian Fairness
## Ethical Framework: Prioritarianism (Parfit)

X/Twitter's algorithm fills 50% of the For You feed with content from accounts the user doesn't follow. It uses SimClusters (145,000 virtual communities) and social graph analysis to select this content.

In practice, **popular accounts dominate** out-of-network recommendations because their content has more engagement signals within clusters.

The **prioritarian ethical constraint** (Derek Parfit) says: benefits to the worse-off should carry more moral weight. A small creator gaining reach matters more than a large creator gaining additional reach.

### The Tension
The algorithm creates a rich-get-richer dynamic. This is the inverse of Parfit's Levelling Down Objection — the system "levels up" the already-powerful.

## Setup

In [2]:
!pip install z3-solver
from z3 import *

'pip' is not recognized as an internal or external command,
operable program or batch file.


## Domain Model

We model 3 creators competing for a fixed pool of out-of-network recommendation slots:
- **Creator 0**: Large influencer (1M followers, quality 75)
- **Creator 1**: Mid-size creator (10K followers, quality 80)
- **Creator 2**: Small creator (500 followers, quality 85)

Note: the small creator has the **best** content quality, but the **lowest** predicted engagement because engagement prediction is biased by existing audience size.

In [3]:
s = Solver()

# Domain: 3 creators
followers            = [Int(f'followers_{i}') for i in range(3)]
content_quality      = [Int(f'content_quality_{i}') for i in range(3)]
predicted_engagement = [Int(f'predicted_engagement_{i}') for i in range(3)]
slots_allocated      = [Int(f'slots_{i}') for i in range(3)]

# Creator 0: Large influencer (1M followers)
s.add(followers[0] == 1000000)
s.add(content_quality[0] == 75)
s.add(predicted_engagement[0] == 90)

# Creator 1: Mid-size creator (10K followers)
s.add(followers[1] == 10000)
s.add(content_quality[1] == 80)
s.add(predicted_engagement[1] == 50)

# Creator 2: Small creator (500 followers)
s.add(followers[2] == 500)
s.add(content_quality[2] == 85)
s.add(predicted_engagement[2] == 20)

# Fixed total slots
TOTAL_SLOTS = 100
s.add(Sum(slots_allocated) == TOTAL_SLOTS)
for i in range(3):
    s.add(slots_allocated[i] >= 0)

print("Scenario: 100 out-of-network slots to allocate")
print("  Creator 0 [LARGE]:  1M followers,  quality=75, pred_engagement=90")
print("  Creator 1 [MID]:    10K followers, quality=80, pred_engagement=50")
print("  Creator 2 [SMALL]:  500 followers, quality=85, pred_engagement=20")

Scenario: 100 out-of-network slots to allocate
  Creator 0 [LARGE]:  1M followers,  quality=75, pred_engagement=90
  Creator 1 [MID]:    10K followers, quality=80, pred_engagement=50
  Creator 2 [SMALL]:  500 followers, quality=85, pred_engagement=20


## Functional Goal: Allocate by Predicted Engagement

The algorithm maximises total engagement by giving more slots to content with higher predicted engagement. This is how SimClusters works: content popular within a cluster gets recommended to other cluster members.

This creates winner-takes-most dynamics where the top creator gets the majority of slots.

In [4]:
# Slots must follow engagement ranking
s.assert_and_track(
    And(
        slots_allocated[0] > slots_allocated[1],
        slots_allocated[1] > slots_allocated[2]
    ),
    "FUNCTIONAL_slots_follow_engagement_ranking"
)

# Top creator gets at least 50% (winner-takes-most)
s.assert_and_track(
    slots_allocated[0] >= 50,
    "FUNCTIONAL_top_creator_majority_slots"
)

print("Functional constraints added:")
print("  - Slots ordered by predicted engagement: large > mid > small")
print("  - Top creator gets >= 50 slots (winner-takes-most)")

Functional constraints added:
  - Slots ordered by predicted engagement: large > mid > small
  - Top creator gets >= 50 slots (winner-takes-most)


## Ethical Goal: Prioritarian Fairness

Parfit's Priority View uses a **concave weighting function**: the moral value of a benefit decreases as the recipient's well-being increases.

$$W_{priority} = \sum_{i=1}^{n} f(wellbeing_i) \quad \text{where } f''(x) < 0$$

We encode this as: when a smaller creator has equal or better content quality than a larger creator, the smaller creator should receive at least as many recommendation slots. This reflects the higher moral weight of benefits to the worse-off.

In [5]:
# Prioritarian: worse-off creators with good content get priority
s.assert_and_track(
    Implies(
        And(content_quality[2] >= content_quality[0],
            followers[2] < followers[0]),
        slots_allocated[2] >= slots_allocated[0]
    ),
    "ETHICAL_prioritise_worse_off_small_vs_large"
)

s.assert_and_track(
    Implies(
        And(content_quality[1] >= content_quality[0],
            followers[1] < followers[0]),
        slots_allocated[1] >= slots_allocated[0]
    ),
    "ETHICAL_prioritise_worse_off_mid_vs_large"
)

# Minimum reach guarantee: no one with quality content gets zero
QUALITY_THRESHOLD = 50
for i in range(3):
    s.assert_and_track(
        Implies(
            content_quality[i] >= QUALITY_THRESHOLD,
            slots_allocated[i] >= 5
        ),
        f"ETHICAL_minimum_reach_guarantee_creator_{i}"
    )

print("Ethical constraints added:")
print("  - Better content + fewer followers → at least as many slots")
print("  - Minimum 5 slots for any creator with quality >= 50")

Ethical constraints added:
  - Better content + fewer followers → at least as many slots
  - Minimum 5 slots for any creator with quality >= 50


## Conflict Detection

In [6]:
print("CONFLICT 3: Out-of-Network Amplification vs. Prioritarian Fairness")
print("Ethical Framework: Prioritarianism (Parfit)")

result = s.check()

if result == sat:
    print("\nResult: SAT (No conflict)")
    m = s.model()
    for i in range(3):
        size = {0: "LARGE (1M)", 1: "MID (10K)", 2: "SMALL (500)"}[i]
        print(f"  Creator {i} [{size}]: quality={m.evaluate(content_quality[i])}, "
              f"pred_engagement={m.evaluate(predicted_engagement[i])}, "
              f"slots={m.evaluate(slots_allocated[i])}")

elif result == unsat:
    print("\nResult: UNSAT — Conflict detected!")
    core = s.unsat_core()
    print(f"\nUnsat Core ({len(core)} conflicting constraints):")
    for c in core:
        print(f"  - {c}")

    print("\nInterpretation:")
    print("  The functional goal (allocate slots by predicted engagement)")
    print("  conflicts with the prioritarian principle (weight benefits")
    print("  towards worse-off creators).")
    print()
    print("  The small creator has the BEST content (quality=85) but gets")
    print("  the FEWEST slots because predicted engagement is lowest.")
    print("  Predicted engagement is biased by existing follower count —")
    print("  a rich-get-richer dynamic that Parfit's framework rejects.")

CONFLICT 3: Out-of-Network Amplification vs. Prioritarian Fairness
Ethical Framework: Prioritarianism (Parfit)

Result: UNSAT — Conflict detected!

Unsat Core (2 conflicting constraints):
  - FUNCTIONAL_slots_follow_engagement_ranking
  - ETHICAL_prioritise_worse_off_mid_vs_large

Interpretation:
  The functional goal (allocate slots by predicted engagement)
  conflicts with the prioritarian principle (weight benefits
  towards worse-off creators).

  The small creator has the BEST content (quality=85) but gets
  the FEWEST slots because predicted engagement is lowest.
  Predicted engagement is biased by existing follower count —
  a rich-get-richer dynamic that Parfit's framework rejects.


## Resolution Options

1. **Quality-first ranking** — weight slot allocation by content quality, not predicted engagement
2. **Concave weighting function** — apply diminishing returns on slots as follower count increases (directly implementing Parfit's mathematical framework)
3. **Affirmative amplification** — reserve a minimum percentage of out-of-network slots for small creators